### Imports

In [ ]:
from __future__ import annotations

import scanpy as sc
import matplotlib.pyplot as plt

import liana as li
from liana.method import (
    cellphonedb, connectome, logfc,
    natmi, singlecellsignalr, geometric_mean
)
import decoupler as dc

from mudata import MuData
from itertools import product

import seaborn as sns
import pandas as pd
from scipy import stats, sparse
import os

# Cell type annotations used in manuscript

## preprocessing

In [ ]:
adata = sc.read_h5ad("/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/data/sc_mosaic_rounded_embeds_NOuncert_subclusters_mofa.h5ad")

In [ ]:
adata.layers["counts"] = adata.X.copy()

# Normalization and log1p
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

adata.raw = adata.copy()

In [ ]:
agg = li.mt.AggregateClass(
    li.mt.aggregate_meta,
    methods=[
        cellphonedb,
        connectome,
        logfc,
        natmi,
        singlecellsignalr,
        geometric_mean
    ]
)

group_key = "cluster_id"

agg(
    adata=adata,
    groupby=group_key,
    resource_name="consensus",
    use_raw=False,
    verbose=True,
)

In [ ]:
factor7_contributing_cells = ['OPC-like', 'OPC', 'TAM-BDM', 'NPC-like', 'MES-like', 'Endothelial', 'Astrocyte', 'AC-like']

In [ ]:
# Filter liana results to just these sender/receiver pairs
liana_df = adata.uns["liana_res"]

# drop autocrine
liana_df = liana_df[liana_df["source"] != liana_df["target"]]

liana_df_subset_factor7 = liana_df[
    (liana_df["source"].isin(factor7_contributing_cells)) &
    (liana_df["target"].isin(factor7_contributing_cells))
]

# Store as a temporary key
adata.uns["liana_res_subset_factor7"] = liana_df_subset_factor7

In [ ]:
custom_palette = {
    'AC-like':      '#1f77b4',  # Blue
    'Astrocyte':    '#d62728',  # Red
    'Endothelial':  '#2ca02c',  # Green
    'MES-like':     '#9467bd',  # Purple
    'NPC-like':     '#8c564b',  # Brown
    'OPC':          '#e377c2',  # Pink
    'OPC-like':     '#ff7f0e',  # Orange
    'TAM-BDM':      '#7f7f7f',  # Gray
}

categories = adata.obs['cluster_id'].cat.categories

new_colors = [0 for i in range(24)]

for cat, color in custom_palette.items():
    if cat in categories:
        # Find the index of the category
        idx = list(categories).index(cat)
        # Update the color at that index
        new_colors[idx] = color
        print(f"Updated {cat} to {color}")

adata.uns['cluster_id_colors'] = new_colors

In [ ]:
adata.write("/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/data/GBmap/ligand_receptor/sc_mosaic_rounded_embeds_NOuncert_subclusters_mofa_ligand_receptor.h5ad")

## Circle plot: Ligand receptor interactions between cell types

In [ ]:
adata = sc.read("/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/data/GBmap/ligand_receptor/sc_mosaic_rounded_embeds_NOuncert_subclusters_mofa_ligand_receptor.h5ad")

In [ ]:
new_color_palette = {
    # Glial/Neural-related cells
    'OPC': '#1f77b4',
    'OPC-like': '#4b9cd3',
    'Oligodendrocyte': '#6baed6',
    'Astrocyte': '#9ecae1',
    'TRAIL+ Astrocytes': '#ff00ff',
    'AC-like': '#6a51a3',
    'NPC-like': '#8c6bb1',
    'RG': '#bcbddc',
    'Neuron': '#dadaeb',

    # Immune cells
    'TAM-MG': '#d62728',
    'TAM-BDM': '#e66767',
    'E-MDSCs': '#8B4513',
    'M-MDSCs': '#DAA520',
    'Mono': '#fdae6b',
    'Neutrophil': '#fdd0a2',
    'DC': '#e31a1c',
    'CD4/CD8': '#fc4e2a',
    'NK': '#feb24c',
    'B cell': '#ffeda0',
    'Plasma B': '#f768a1',
    'Mast': '#ae017e',

    # Vascular/Other
    'Endothelial': '#31a354',
    'Mural cell': '#74c476',
    'MES-like': '#bae4b3'
}

old_palette_keys = [
    'AC-like', 'Astrocyte', 'Endothelial', 'MES-like', 
    'NPC-like', 'OPC', 'OPC-like', 'TAM-BDM'
]

categories = adata.obs['cluster_id'].cat.categories
# Ensure we are working with a mutable list copy of the current colors
current_colors = list(adata.uns['cluster_id_colors'])

for cell_type in old_palette_keys:
    # Check if the cell type exists in your data AND in the new palette
    if cell_type in categories and cell_type in new_color_palette:
        # Find the index of the category
        idx = list(categories).index(cell_type)
        
        # Update with the NEW color
        new_color = new_color_palette[cell_type]
        current_colors[idx] = new_color
        
        print(f"Updated {cell_type} (index {idx}) to new color: {new_color}")

adata.uns['cluster_id_colors'] = current_colors

In [ ]:
li.pl.circle_plot(
    adata=adata,
    groupby="cluster_id",      # your cell-type column
    score_key="magnitude_rank",            # which score to size edges by
    inverse_score=True,                    # so bigger = stronger interaction
    pivot_mode="counts",                   # count how many L–R pairs per edge
    filter_fun=lambda df: df["specificity_rank"] <= 0.05,  # only keep top specificity
    figure_size=(8, 8),
    uns_key="liana_res_subset_factor7"      # where results live
)

plt.savefig("/home/sagemaker-user/user-default-efs/factor_7_circleplot_NOuncert_subclusters_mofa_new_colors.png")

## Dotplot: ligand receptor interactions

In [ ]:
source_cells = ['OPC-like', 'AC-like', 'NPC-like', 'MES-like']
target_cells = ['OPC', 'Astrocyte', 'TAM-BDM', 'Endothelial']

In [ ]:
#fig = li.pl.dotplot(
li.pl.dotplot(
    adata=adata,
    colour="magnitude_rank",
    size="specificity_rank",
    inverse_colour=True,
    inverse_size=True,
    source_labels=source_cells,
    target_labels=target_cells,
    top_n=5,
    orderby="magnitude_rank",
    orderby_ascending=True,
    figure_size=(16, 8)
)

#fig.save(
#    filename="/home/sagemaker-user/user-default-efs/factor_7_dotplot_malignant_as_source.png",
#    dpi=300,
#    verbose=False,
#    bbox_inches='tight'
#)

In [ ]:
liana_df = adata.uns["liana_res"].copy()
liana_df_subset = liana_df[
    (liana_df["source"].isin(source_cells)) &
    (liana_df["target"].isin(target_cells))
]

adata.uns["liana_res_factor7_filtered"] = liana_df_subset

fig = li.pl.dotplot(
    adata=adata,
    uns_key="liana_res_factor7_filtered",
    colour="magnitude_rank",
    size="specificity_rank",
    inverse_colour=True,
    inverse_size=True,
    source_labels=source_cells,
    target_labels=target_cells,
    top_n=5,
    orderby="magnitude_rank",
    orderby_ascending=True, 
    figure_size=(10, 6)
)

fig.save(
    filename="/home/sagemaker-user/user-default-efs/factor_7_dotplot_malignant_as_source_top5.png",
    dpi=300,
    verbose=False,
    bbox_inches='tight'
)

## analysis about LR interaction per sample 

In [ ]:
adata = sc.read_h5ad("/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/data/GBmap/ligand_receptor/sc_mosaic_rounded_embeds_NOuncert_subclusters_mofa_ligand_receptor.h5ad")

In [ ]:
# Retrieve the results DataFrame
liana_df = adata.uns["liana_res_subset_factor7"].copy()

# Sort by magnitude_rank (Ascending)
top_interactions = liana_df.sort_values("magnitude_rank", ascending=True).head(10)

# Create the list of tuples automatically
# Format: (Ligand, Receptor, Source Cell, Target Cell)
pairs_to_plot = list(zip(
    top_interactions["ligand_complex"], 
    top_interactions["receptor_complex"], 
    top_interactions["source"], 
    top_interactions["target"]
))

print("--- Top 10 Unbiased Interactions ---")
for p in pairs_to_plot:
    print(f"Interaction: {p[0]} - {p[1]}")
    print(f"Direction:   {p[2]} -> {p[3]}")
    print("---")

In [ ]:

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
sns.set_theme(style="ticks", context="paper", font_scale=1.1)


SAMPLE_COL = 'sample_id' 
CELL_TYPE_COL = 'cluster_id'

def get_sample_means(ad_obj, gene, s_key):
    """
    Calculates the mean expression of a gene per sample, handling both 
    sparse and dense matrices correctly.
    """
    # Extract expression vector
    expr_values = ad_obj[:, gene].X
    
    # Handle sparse vs dense
    if sparse.issparse(expr_values):
        expr_values = expr_values.toarray().flatten()
    else:
        expr_values = expr_values.flatten()
        
    # Create temporary DF for aggregation
    exp_df = pd.DataFrame({'expr': expr_values}, index=ad_obj.obs.index)
    exp_df['sample'] = ad_obj.obs[s_key].values
    
    # Return mean expression per sample
    return exp_df.groupby('sample')['expr'].mean()

def plot_lr_correlation_pub(ax, adata, ligand, receptor, source_cell, target_cell, 
                            sample_key, cell_type_key='cluster_id'):
    """
    Plots correlation on a specific matplotlib axis (ax) with publication styling.
    """
    source_adata = adata[adata.obs[cell_type_key] == source_cell]
    target_adata = adata[adata.obs[cell_type_key] == target_cell]
    
    # Basic validation
    if source_adata.n_obs == 0:
        ax.text(0.5, 0.5, f"No cells found:\n{source_cell}", ha='center', va='center')
        return
    if target_adata.n_obs == 0:
        ax.text(0.5, 0.5, f"No cells found:\n{target_cell}", ha='center', va='center')
        return
    if ligand not in source_adata.var_names:
        ax.text(0.5, 0.5, f"Gene missing:\n{ligand}", ha='center', va='center')
        return
    if receptor not in target_adata.var_names:
        ax.text(0.5, 0.5, f"Gene missing:\n{receptor}", ha='center', va='center')
        return

    try:
        source_means = get_sample_means(source_adata, ligand, sample_key)
        target_means = get_sample_means(target_adata, receptor, sample_key)
    except Exception as e:
        ax.text(0.5, 0.5, f"Error calculating means:\n{str(e)}", ha='center', va='center')
        return
    
    # Merge (inner join)
    x_label = f'{ligand} in {source_cell}'
    y_label = f'{receptor} in {target_cell}'
    plot_df = pd.DataFrame({
        x_label: source_means,
        y_label: target_means
    }).dropna()
    
    # Check sample size
    if len(plot_df) < 5:
        ax.text(0.5, 0.5, "Not enough samples\nfor correlation (n<5)", ha='center', va='center')
        return
    
    r, p = stats.pearsonr(plot_df[x_label], plot_df[y_label])
    p_text = f"{p:.2e}" if p < 0.001 else f"{p:.3f}"
    stats_text = f"$R$ = {r:.2f}\n$p$ = {p_text}\n$n$ = {len(plot_df)}"

    color_point = "#34495e" # Dark slate grey
    color_line = "#c0392b"  # Deep red

    # Scatter plot with regression line
    sns.regplot(data=plot_df, x=x_label, y=y_label, ax=ax,
                scatter_kws={'s': 50, 'alpha': 0.7, 'color': color_point, 
                             'edgecolor': 'white', 'linewidths': 0.5},
                line_kws={'color': color_line, 'linewidth': 2},
                ci=95) 

    # Add stats box
    ax.annotate(stats_text, xy=(0.05, 0.95), xycoords='axes fraction',
                ha='left', va='top', fontsize=9,
                bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="lightgray", lw=1, alpha=0.9))
    
    # Formatting
    ax.set_xlabel(x_label, fontsize=10, fontweight='medium')
    ax.set_ylabel(y_label, fontsize=10, fontweight='medium')
    sns.despine(ax=ax, offset=5)


output_dir = "figures"
os.makedirs(output_dir, exist_ok=True)

# Manually selected from Top 10 to maximize biological coverage
pairs_to_plot = [
    # 1. Strongest Malignant -> Myeloid signal (AC-like)
    ('APP', 'CD74', 'AC-like', 'TAM-BDM'),
    
    # 2. Mesenchymal -> Myeloid signal (shows robustness of APP-CD74 axis)
    ('APP', 'CD74', 'MES-like', 'TAM-BDM'),
    
    # 3. Strongest PTPRZ1 axis signal (AC-like)
    ('PTN', 'PTPRZ1', 'AC-like', 'OPC'),
    
    # 4. Progenitor-specific PTPRZ1 signal (OPC-like)
    ('PTN', 'PTPRZ1', 'OPC-like', 'OPC'),
]

# Initialize Figure (2x2 Grid)
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
axes = axes.flatten()

print(f"Generating plots using sample column: '{SAMPLE_COL}'...")

for i, (lig, rec, src, tgt) in enumerate(pairs_to_plot):
    if i < len(axes):
        plot_lr_correlation_pub(
            axes[i], adata, lig, rec, src, tgt, 
            sample_key=SAMPLE_COL, 
            cell_type_key=CELL_TYPE_COL
        )
        # Add panel label (A, B, C, D)
        axes[i].text(-0.15, 1.05, chr(65+i), transform=axes[i].transAxes, 
                     fontsize=16, fontweight='bold', va='top', ha='right')

plt.tight_layout()
output_path = os.path.join(output_dir, "SuppFig_LR_Scatter_Validation.pdf")
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Done! Figure saved to: {output_path}")

plt.show()

## spatial: cell type - transcriptions factor interactions

In [ ]:
adata = sc.read_h5ad("/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/downstream/v2_visium_concat_deconvolved_pathways_patho_hoptimus1.h5ad")

In [ ]:
cols = adata.obs.columns
start = cols.get_loc("AC-like")
end = cols.get_loc("TRAIL+ Astrocytes") # inclusive
celltype_cols = cols[start:end+1].tolist()

print(f"{len(celltype_cols)} cell types:", celltype_cols)

In [ ]:
sid = "HKG002a"

adata_sample = adata[adata.obs["sample_id"] == sid].copy()

In [ ]:
comps = li.ut.obsm_to_adata(adata_sample, 'q05_cell_abundance_w_sf')

# check key cell types
sc.pl.spatial(comps,
              color=['AC-like', 'Astrocyte', 'B cell', 'CD4/CD8', 'DC', 'E-MDSCs',
                     'Endothelial', 'M-MDSCs', 'MES-like', 'Mast', 'Mono', 'Mural cell',
                     'NK', 'NPC-like', 'Neuron', 'Neutrophil', 'OPC', 'OPC-like',
                     'Oligodendrocyte', 'Plasma B', 'RG', 'TAM-BDM', 'TAM-MG', 'TRAIL+ Astrocytes'],
              size=1.3,
              ncols=4,
              spot_size=130
             )

In [ ]:
# ! only for sample 028a due to missing cell types here !
cell_types_to_exclude = ['B cell', 'NPC-like', 'Neuron', 'RG', 'TRAIL+ Astrocytes']

comps = li.ut.obsm_to_adata(adata_sample, 'q05_cell_abundance_w_sf')

prefix = 'q05cell_abundance_w_sf_'

prefixed_to_exclude = [prefix + ct for ct in cell_types_to_exclude]

comps_to_keep_prefixed = comps.var_names[~comps.var_names.isin(prefixed_to_exclude)]
comps = comps[:, comps_to_keep_prefixed].copy()

print(f"Cell types remaining in comps.X (prefixed): {comps.var_names.tolist()}")

remaining_pure_cell_types = [
    name.replace(prefix, '', 1) 
    for name in comps.var_names.tolist()
]

cols_to_drop = [col for col in remaining_pure_cell_types if col in comps.obs.columns]

if cols_to_drop:
    print(f"Removing redundant cell type columns from comps.obs: {cols_to_drop}")
    comps.obs = comps.obs.drop(columns=cols_to_drop)
else:
    print("No redundant cell type columns found in comps.obs to drop.")

In [ ]:
# Get transcription factor resource
net = dc.get_collectri()

In [ ]:
# run enrichment
dc.run_ulm(adata_sample, net=net, use_raw=False, verbose=True)

In [ ]:
est = li.ut.obsm_to_adata(adata_sample, 'ulm_estimate')
est.var['cv'] =  est.X.std(axis=0) / est.X.mean(axis=0)
top_tfs = est.var.sort_values('cv', ascending=False, key=abs).head(50).index

In [ ]:
mdata = MuData({"tf":est, "comps":comps})
mdata.obsp = adata_sample.obsp
mdata.uns = adata_sample.uns
mdata.obsm = adata_sample.obsm

li.ut.spatial_neighbors(
    mdata,
    bandwidth=200,
    cutoff=0.1,
    kernel="gaussian",
    set_diag=True
)

In [ ]:
interactions = list(product(comps.var.index, top_tfs))

In [ ]:
bdata = li.mt.bivariate(mdata,
                        x_mod="comps",
                        y_mod="tf",
                        x_transform=sc.pp.scale,
                        y_transform=sc.pp.scale,
                        local_name="cosine",
                        interactions=interactions,
                        mask_negatives=True,
                        add_categories=True,
                        x_use_raw=False,
                        y_use_raw=False,
                        xy_sep="<->",
                        x_name='celltype',
                        y_name='tf'
                        )

In [ ]:
# Clean variable names in bdata.var
bdata.var.index = bdata.var.index.str.replace("q05cell_abundance_w_sf_", "", regex=False)

In [ ]:
bdata.var.sort_values("mean", ascending=False).head(20)

In [ ]:
ax = sc.pl.spatial(bdata,
              color=['MES-like<->TRPS1'],
              size=1.4,
              cmap="Reds",
              vmax=1,
              vmin=0,
              spot_size=130,
              save='_spatial_LR_MESlike_TRPS1_HKG002a'
             )

In [ ]:
sc.pl.spatial(mdata.mod['tf'],
              color=['TRPS1'],
              size=1.4,
              vcenter=0,
              cmap='RdBu_r',
              spot_size=110,
              save='_spatial_TF_activity_TRPS1_HKG002a'
              )